In [1]:
from langgraph.graph import StateGraph
import sys
sys.path.append('/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach')
from typing import TypedDict
import os 
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from app.chat_memory import get_chat_history
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import HumanMessage
from app.video_processing import analyze_video
from vectorstore_setup import clean_classification_text
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langgraph.graph import StateGraph, START, END
from app.chat_memory import get_chat_history 
import tempfile

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
vectorstore = Chroma(persist_directory="/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/chroma_db", embedding_function=embeddings)

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGSMITH_API_KEY")


# Whisper speach to text 
from openai import OpenAI
client = OpenAI()




In [2]:
query_agent = ChatPromptTemplate.from_messages([
    ("system", """Your job is to classify user queries into two buckets: 
     - memory
     - memory and vectorstore
     
     *Guidelines*
     Questions regarding exercises, exercise form, or technique: vectorstore & memory
     General question or statements: memory
   
     *Respond to queries with only the correct class*
     Examples: 
     User: how should I grip the bar for bench press?
     Agent: vectorstore & memory

     User: what muscles should I feel during the Romanian dead lift?
     Agent: vectorstore & memory

     User: How's this? 
     Agent: vectorstore & memory

     User: Is this good squat technique? 
     Agent: vectorstore & memory

     User: Was my bar path better?
     Agent: vectorstore & memory

     User: thakns for your help!
     Agent: memory

"""),

     ("human", "{query}")
     
])

llm = ChatOpenAI(model='gpt-5.4-nano')

output_parser = StrOutputParser()

router_chain = (query_agent | llm | output_parser).with_config({"run_name": "router_agent"})


In [3]:
# Define the state

# the state is a dictionary

class GraphState(TypedDict):

    session_id: int 

    user_query: str

    user_video: str

    classification_image: str

    classified_keywords: str

    top_k_chunks: str

    encoded_images: list[str]

    response: str

# Router function: conditional edge

In [4]:
def route_query(state: GraphState):
    # first check: is there a video?
    if state["user_video"]:
        return "video_encoder"
    
    # no video — ask the router_prompt LLM which path
    route = router_chain.invoke({"query": state["user_query"]})
    route = route.lower().strip()
    
    if "vectorstore" in route:
        return "vector_db"
    else:
        return "chat_memory"

# Video Encoder Node

In [5]:
def video_encoder_node(state: GraphState):
    user_video = state["user_video"]
    user_video_encoded = analyze_video(filepath_in=user_video, frame_count=15, max_seconds=10) # encodes video
    encoded_images = [] # creates a list of encoded images for LLM
    for images in user_video_encoded:
        encoded_images.append({"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{images}"}})
        classification_image = user_video_encoded[0] 
    
    return {"classification_image": classification_image, "encoded_images": encoded_images}


# Video Classification Node

In [6]:
# video classification router NODE

def video_classification_node(state: GraphState):
    classification_image = state["classification_image"]

    router_llm = ChatOpenAI(model='gpt-5.4-nano', temperature=0.0)

    response = router_llm.invoke([

    {"role": "user", "content": 

    [{"type": "text", "text":  """Your job is to analyze images of users working out for proper form, and list the key checkpoints of their to body evaluate. 
    Give me ONLY the bodypart checkpoints. Do NOT include evaluation suggestions. Do NOT include an intro sentence. 
    Output format should be exactly the example below.
    **Example**
    Overhead press

    1. Feet & base
    2. Glutes & legs
    3. Core & Ribcage
    4. Shoulder position
    5. Bar path
    6. Head & Neck
    7. Lockout position
    8. Tempo and control
    """},


        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{classification_image}"}}
        ]}


    ])

    classified_keywords = clean_classification_text(response)

    return {"classified_keywords": classified_keywords}


# VectorDB Node

In [7]:
def vector_db_node(state: GraphState):
    user_query = state["user_query"]
    classified_keywords = state.get("classified_keywords", "")
    retrieval_query = classified_keywords

    if classified_keywords:
        results_user = vectorstore.similarity_search(user_query, k=5)
        results_video = vectorstore.similarity_search(retrieval_query, k=5)
        results = results_user + results_video

    if not classified_keywords: 
        results_user = vectorstore.similarity_search(user_query, k=5)
        results = results_user

    unique_results = []
    seen = set()

    for r in results:
        content = r.page_content
        if content not in seen:
            seen.add(content)
            unique_results.append(r)


    top_k_chunks = "\n".join([r.page_content for r in unique_results])

    print(f"RESULTS COUNT: {len(results)}")
    print(f"UNIQUE RESULTS COUNT: {len(unique_results)}")
    for r in unique_results[:2]:
        print(f"CHUNK PREVIEW: {r.page_content[:100]}")

    return {"top_k_chunks": top_k_chunks}

# Response Generator Node

In [8]:
def response_generator(state: GraphState):
    top_k_chunks = state['top_k_chunks']
    encoded_images = state.get("encoded_images", [])
    session_id = state["session_id"]
    user_query = state["user_query"]
    classified_keywords = state.get("classified_keywords", "")

    response_generator_prompt = ChatPromptTemplate.from_messages([
            ("system", """You are a world-class fitness coach. You have extensive experience in helping weight lifters achieve perfect form and maximum hypertrophy. 
    Your job is to analyze images of users lifting weights, offer them advice from your context, and to answer any questions they might have. 
    Look for issues related to form, safety, and unhelpful camera angles.
    Be specific about what you observe and include that in your feedback.
    Do not make any mention to "frames". To the user, you are watching a video.
             
             
    # HOW TO RESPOND

    Decide which mode you're in based on the user's message:

    **Mode A — Initial video analysis**:

    Analyze the lift in this fixed order. For each section, either identify a specific issue or state the section looks correct:

    1. Setup and starting position
    2. Lifting phase (concentric) — the effort portion
    3. Lowering phase (eccentric) — the controlled return
    4. Range of motion and completion — did the rep finish at full range, top and bottom?

    After the structured analysis, end with ONE offer for a next step (e.g., "If you want, I can also...").

    If the frames don't clearly show a phase, say so rather than guessing.

    **Mode B — Follow-up question** (user asks a specific question, e.g. "why does X matter?", "how do I fix Y?", "what about Z?"):
    - Answer the question directly using the context below
    - Do NOT re-describe the video or repeat your earlier analysis
    - Do NOT restart with "What I'm seeing in your bench" or similar
    - Reference the user's specific lift only if it's directly relevant to the answer
    - Keep it focused and conversational
    - Maximum 150 words

    # TONE
    Eager and excited to help.
             
    # CONTEXT
    Use ONLY the following context for coaching advice:
    {top_k_chunks}
    
    If the query or image isn't in context, reply, 'I don't have expert coaching advice for this exercise yet. 
    Currently I can analyze: bench press, overhead press, incline bench press...'"
        
    ---  


    """),
        MessagesPlaceholder(variable_name="history"),
        MessagesPlaceholder(variable_name="input"),
    ])

    user_msg = HumanMessage(content=[
        {"type": "text", "text": user_query},
        *encoded_images,   # <- your list of {"type":"image_url",...}
    ])

    response_generator_llm = ChatOpenAI(model='gpt-5.4',
                    temperature=0.5)

    response_generator_output_parser = StrOutputParser()

    response_generator_chain = response_generator_prompt | response_generator_llm | response_generator_output_parser


    message_history = get_chat_history(session_id)

    fitness_analysis = response_generator_chain.invoke(
        {   "input": [user_msg], 
            'history': message_history.messages, 
            "top_k_chunks": top_k_chunks, 
            "classified_keywords": classified_keywords}
            )


    memory_summary_prompt = ChatPromptTemplate.from_messages([
        ("system", """
    Summarize this fitness analysis for future follow-up questions.
    Keep only facts needed for chat continuity.

    Return:
    - exercise
    - main issues
    - priority fixes
    - important uncertainties
    - concise coaching summary

    Maximum 150 words.
    """),
        ("human", "{fitness_analysis}")
    ])

    memory_summary_llm = ChatOpenAI(model='gpt-5.4-nano')
    memory_summary_output_parser = StrOutputParser()
    memory_summary_chain = memory_summary_prompt | memory_summary_llm | memory_summary_output_parser

    summarized_response = memory_summary_chain.invoke({"fitness_analysis": fitness_analysis})
    message_history.add_user_message(user_query) # save user query to memory
    message_history.add_ai_message(summarized_response) # save summarized response to memory


    return {"response": fitness_analysis} # return full analysis to user
     
    

# Chat Memory node

In [9]:
# module level - built once when file loads 
chat_memory_prompt = ChatPromptTemplate.from_messages([
            ("system", """You are a world-class fitness coach. 
             You have extensive experience in helping weight lifters achieve perfect form and maximum hypertrophy. 
    Your job is to analyze images of users lifting weights, offer them advice from your context, and to answer any questions they might have. 


    """),
        MessagesPlaceholder(variable_name="history"),
        MessagesPlaceholder(variable_name="input"),
    ])

   
chat_memory_llm = ChatOpenAI(model='gpt-5.4-nano', temperature=0.5)
chat_memory_output_parser = StrOutputParser()

chat_memory_chain = chat_memory_prompt | chat_memory_llm | chat_memory_output_parser

chat_memory_with_history = RunnableWithMessageHistory(
    chat_memory_chain,
    get_session_history=get_chat_history,
    input_messages_key="input",
    history_messages_key="history"

)

# Function: calls only what changes
def chat_memory(state: GraphState):
    user_query = state["user_query"]
    session_id = state["session_id"]

    user_msg = HumanMessage(content=[
        {"type": "text", "text": user_query}
    ])

    response = chat_memory_with_history.invoke(
    {"input": [user_msg]},
    config={"configurable": {"session_id": session_id}}
)
    return {"response": response}
    

# Audio transcription function

In [10]:
# def transcribe_audio(audio_path):
#     audio_file = open(audio_path, "rb") # rb = read binary. audio files are binary data (raw bytes). this tells python gto open as binary instaed of text
#     transcription = client.audio.transcriptions.create(
#         model="whisper-1",
#         file=audio_file
#     )
# def transcribe_audio(audio_path):
#     audio_file = open(audio_path, "rb") # rb = read binary. audio files are binary data (raw bytes). this tells python gto open as binary instaed of text
#     transcription = client.audio.transcriptions.create(
#         model="whisper-1",
#         file=audio_file
#     )

#     return transcription.text
    

# user_query = transcribe_audio("/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/data/raw_voice_notes/use_your_lats.m4a")
# print(user_query)

In [11]:
# rebuild graph fresh each time this cell runs
workflow = StateGraph(GraphState)

# conditional edge
workflow.add_conditional_edges(START, route_query)

# add nodes
workflow.add_node("video_encoder", video_encoder_node)
workflow.add_node("video_classification_router", video_classification_node)
workflow.add_node("vector_db", vector_db_node)
workflow.add_node("response_generator", response_generator)
workflow.add_node("chat_memory", chat_memory)



# connect them
workflow.add_edge("chat_memory", END)
workflow.add_edge("video_encoder", "video_classification_router")
workflow.add_edge("video_classification_router", "vector_db")
workflow.add_edge("vector_db", "response_generator")
workflow.add_edge("response_generator", END)

app = workflow.compile()

In [12]:
# result = app.invoke({
#     "session_id": 115,
#     "user_query": "how is my form here?",
#     "user_video": "/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/data/raw_workout_videos/bench_press/bench_14.mp4"
# })

# print(result["response"])

In [13]:

# result = app.invoke({
#     "session_id": 114,
#     "user_query": user_query,
#     "user_video": ""
# })

# print(result["response"])

In [14]:

# result = app.invoke({
#     "session_id": 115,
#     "user_query": "what did you say about foot pressure?",
#     "user_video": ""
# })

# print(result["response"])

In [15]:

# result = app.invoke({
#     "session_id": 115,
#     "user_query": "thanks!",
#     "user_video": ""
# })

# print(result["response"])

In [16]:
def correctness_evaluator(run, example):
    generated = run.outputs["answer"]
    reference = example.outputs["answer"]
    question = example.inputs["user_query"]

    prompt = f"""You are evaluating a fitness coaching AI assistant.
Your job is to assess the quality of its response against a reference answer. 
The reference answer contains the minimum key observations. The generated answer may expand on these with additional coaching advice and this should not be penalized.

Question: {question}
Reference answer: {reference}
Generated answer: {generated}

Scoring criteria:

Score 0 — The generated answer contains incorrect or potentially harmful advice that contradicts the reference answer. This is the worst outcome.
Score 1 — The generated answer is incomplete or omits important details from the reference, but contains nothing incorrect or harmful.
Score 2 — The generated answer is correct and captures the key facts from the reference answer.

Important: Penalize incorrect advice more harshly than omission. A response that says nothing is better than one that says something wrong.

Reply with only the number: 0, 1, or 2."""

    from openai import OpenAI
    openai_client = OpenAI()
    result = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}]
    )
    try: 
        score_raw = result.choices[0].message.content.strip()
        score = int(''.join(filter(str.isdigit, score_raw))[0])
    except Exception as e:
        print(f"Error parsing score: {e}, raw output: {score_raw}")
        score = -1
    return {"key": "correctness", "score": score / 2} # normalize the score for LangSmith configurations

In [17]:
def hallucination_evaluator(run, example):
    generated = run.outputs["answer"]
    context = run.outputs["context"]
    
    prompt = f"""YYou are evaluating a fitness coaching AI assistant for hallucinations.

This AI analyzes workout VIDEOS and uses retrieved coaching context to provide form advice. 
The AI has two information sources:
1. Retrieved text context (coaching knowledge)
2. Video frames of the user's actual lift (visual observations)

<Instructions>
  - Claims about coaching best practices (e.g. "elbows should be 45-60 degrees") should be verified against the retrieved context
  - Claims about what the user is DOING in their video (e.g. "your elbows are flaring wide") are visual observations and should NOT be penalized for not appearing in the retrieved context
  - Only flag a claim as hallucinated if it contradicts the retrieved context or makes up coaching advice not supported by the context
  - Observations about the user's form that are plausible given a workout video should be treated as valid even if not in the context
</Instructions>

Retrieved context:
{context}

Generated answer:
{generated}

Scoring criteria:

Score 0 — The answer contains coaching advice that directly contradicts the retrieved context, or invents training recommendations with no basis in the context. This is the worst outcome.
Score 1 — The coaching advice is mostly supported by the context but includes minor unsupported recommendations.
Score 2 — All coaching advice and recommendations are directly supported by the retrieved context. Visual observations about the user's form do not count against this score.

Reply with only the number: 0, 1, or 2."""

    from openai import OpenAI
    openai_client = OpenAI()
    result = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}]
    )
    try:
        score_raw = result.choices[0].message.content.strip()
        score = int(''.join(filter(str.isdigit, score_raw))[0])
    except:
        score = None
    return {"key": "hallucination", "score": score / 2} # normalize the score

In [18]:
from uuid import uuid4

def run_rag_pipeline(inputs: dict, attachments: dict) -> dict:
    # grab video from attachments in LangSmith
    video_name = list(attachments.keys())[0] 
    # read the video as raw bytes
    video_bytes = attachments[video_name]["reader"].read()
    
    # save to temp file because analyze_video expects a filepath
    with tempfile.NamedTemporaryFile(suffix=".mp4", delete=False) as tmp:
        # . delete=False means "don't delete this file when we're done writing" because we still need it for the pipeline.
        tmp.write(video_bytes)
        tmp_path = tmp.name
    
    state = {
        "user_query": inputs["user_query"],
        "session_id": f"eval-{uuid4()}",
        "classified_keywords": "",
        "encoded_images": [],
        "user_video": tmp_path
    }
    
    state = {**state, **video_encoder_node(state)}
    state = {**state, **video_classification_node(state)}
    print(f"BEFORE VECTOR DB - classified_keywords: {state.get('classified_keywords', 'MISSING')}")
    state = {**state, **vector_db_node(state)}
    print(f"AFTER VECTOR DB - top_k_chunks length: {len(state.get('top_k_chunks', ''))}")
    state = {**state, **response_generator(state)}

    print("STATE KEYS:", state.keys())
    print("RESPONSE:", state.get("response", "NOT FOUND"))
    
    return {"answer": state["response"],
            "context": state["top_k_chunks"]}

In [ ]:
from langsmith import Client
from langsmith.evaluation import evaluate

langsmith_client = Client()

os.environ["LANGCHAIN_TRACING_V2"] = "false" 
# stops LangSmith from trying to log all the big intermediate data
# evaluate() function will still capture your inputs, outputs, and evaluator scores
# it just won't try to upload the full trace with all those heavy frames.
# 
results = evaluate(
    run_rag_pipeline,
    data="Image assessment accuracy v1 — variance subset (5)",
    evaluators=[correctness_evaluator, hallucination_evaluator],
    experiment_prefix="test02-system-prompt-scaffold-update",
    num_repetitions=10,
)

View the evaluation results for experiment: 'test02-system-prompt-scaffold-update-7f3f5559' at:
https://smith.langchain.com/o/5dffb1fc-82e5-4000-a9a5-d9c923e0f73c/datasets/d8117071-d366-420a-ae45-b8b56c0be8fb/compare?selectedSessions=2ce1d951-dd09-4250-8b5c-462aa10073a8




0it [00:00, ?it/s]

Frames processed: 15, (10s cap)
Saved to /var/folders/2n/c95ntf5d1wx1_vrd1b3ljbkw0000gn/T/tmpksdmet19/tmp78n5jup9
Video name: tmp78n5jup9
BEFORE VECTOR DB - classified_keywords: Bench press     Setup   grip width    Wrist position   forearm angle    Shoulder position  scapula set     Elbow tracking    Bar height over chest    Chest   upper torso alignment    Core bracing   ribcage position    Lower body contact  feet planted stance     Back arch   glute engagement     Bar path  bottom position to press 
RESULTS COUNT: 10
UNIQUE RESULTS COUNT: 10
CHUNK PREVIEW: Now, your back should be planted on the bench, with your eyes directly under the barbell. Position y
CHUNK PREVIEW: along the bench through the hips into the arched back to reinforce the arch and keep the chest in it
AFTER VECTOR DB - top_k_chunks length: 8884
STATE KEYS: dict_keys(['user_query', 'session_id', 'classified_keywords', 'encoded_images', 'user_video', 'classification_image', 'top_k_chunks', 'response'])
RESPONSE: You

In [20]:
# from langsmith import Client
# from langsmith.evaluation import evaluate

# langsmith_client = Client()

# os.environ["LANGCHAIN_TRACING_V2"] = "false" 
# # stops LangSmith from trying to log all the big intermediate data
# # evaluate() function will still capture your inputs, outputs, and evaluator scores
# # it just won't try to upload the full trace with all those heavy frames.
# # 
# results = evaluate(
#     run_rag_pipeline,
#     data="Image assessment response accuracy v1",
#     evaluators=[correctness_evaluator, hallucination_evaluator],
#     experiment_prefix="test43-system-prompt-scaffold-update",
# )